## Implementing MLP Architecture

An MLP is a neural network made of fully connected layers:

```text
Input → Hidden Layer(s) → Output
```

MLPs expect a **2D tensor** as input:

```text
(Batch Size, Features)
```

However, images are usually **3D tensors**:

```text
(Channel, Height, Width)
```

and batches of images are **4D tensors**:

```text
(Batch, Channel, Height, Width)
```

Therefore, before feeding images into an MLP, we must **flatten** them:

```text
(32, 3, 224, 224) → (32, 150528)
```

This conversion allows the MLP to perform matrix multiplication on the input.

In [1]:
import numpy as np

class MLP:

    def __init__(self, in_dim, out_dim):

        # --------------------------------------------------
        # Weight Matrix
        #
        # Shape:
        # (Input Features, Output Features)
        #
        # Here:
        # (300, 1)
        # --------------------------------------------------
        self.w = np.random.rand(in_dim, out_dim)

        # --------------------------------------------------
        # Bias Vector
        #
        # Shape:
        # (Output Features, 1)
        # --------------------------------------------------
        self.b = np.random.rand(out_dim, 1)

        print(f"Weight Shape = {self.w.shape}")
        print(f"Bias Shape   = {self.b.shape}")

    def forward(self, x):

        print("\nInside Forward Pass")
        print(f"Input Shape = {x.shape}")

        # --------------------------------------------------
        # Matrix Multiplication
        #
        # (1, 300) @ (300, 1)
        #
        # Result:
        # (1, 1)
        # --------------------------------------------------
        out = x @ self.w

        print(f"After x @ W = {out.shape}")

        out = out + self.b.T

        print(f"After Adding Bias = {out.shape}")

        return out


# --------------------------------------------------
# Create a fake RGB image
#
# Shape:
# (Channels, Height, Width)
# --------------------------------------------------
height = 10
width = 10

img = np.random.rand(3, height, width)

print("=" * 60)
print("ORIGINAL IMAGE")
print("=" * 60)

print(f"Image Shape = {img.shape}")

print("\nHow to read (3, 10, 10)?")
print("3 Channels")
print("10 Height")
print("10 Width")

# --------------------------------------------------
# MLP expects:
#
# (Batch Size, Features)
#
# Therefore:
#
# (3, 10, 10)
#
# must become:
#
# (1, 300)
#
# because:
#
# 3 × 10 × 10 = 300
# --------------------------------------------------
print("\nMLP expects a 2D tensor:")
print("(Batch Size, Features)")

print("\nFlattening Image...")

flattened_img = img.reshape(1, -1)

print(f"New Shape = {flattened_img.shape}")

print("\nHow to read (1, 300)?")
print("1 Image")
print("300 Features")

# --------------------------------------------------
# Create MLP
# --------------------------------------------------
mlp = MLP(
    in_dim=3 * height * width,
    out_dim=1
)

# --------------------------------------------------
# Forward Pass
# --------------------------------------------------
prediction = mlp.forward(flattened_img)

print("\nFinal Output")
print(prediction)

print(f"Output Shape = {prediction.shape}")

print("\nKey Lesson:")
print("Images are naturally 3D tensors.")
print("MLPs require 2D tensors.")
print("Therefore we flatten:")
print("(3, 10, 10) -> (1, 300)")
print("before feeding the image into the network.")

ORIGINAL IMAGE
Image Shape = (3, 10, 10)

How to read (3, 10, 10)?
3 Channels
10 Height
10 Width

MLP expects a 2D tensor:
(Batch Size, Features)

Flattening Image...
New Shape = (1, 300)

How to read (1, 300)?
1 Image
300 Features
Weight Shape = (300, 1)
Bias Shape   = (1, 1)

Inside Forward Pass
Input Shape = (1, 300)
After x @ W = (1, 1)
After Adding Bias = (1, 1)

Final Output
[[67.54157658]]
Output Shape = (1, 1)

Key Lesson:
Images are naturally 3D tensors.
MLPs require 2D tensors.
Therefore we flatten:
(3, 10, 10) -> (1, 300)
before feeding the image into the network.


## Batch Processing vs. Sequential Iteration

When deploying computer vision pipelines, feeding images individually requires multiple sequential forward passes, creating a massive computing bottleneck. 

Instead, we group data into an **$N$-dimensional batch tensor `[N, C, H, W]`**. This leverages the underlying GPU vectorization engine to process all images in a single parallelized matrix operation, maximizing memory bandwidth and optimizing training throughput.

In [2]:
import numpy as np

# Simulate 5 separate raw RGB image matrices (Height=64, Width=64)
img_1 = np.random.rand(3, 64, 64) 
img_2 = np.random.rand(3, 64, 64) 
img_3 = np.random.rand(3, 64, 64) 
img_4 = np.random.rand(3, 64, 64) 
img_5 = np.random.rand(3, 64, 64) 

print(f"Single image tensor layout shape: {img_1.shape}")

Single image tensor layout shape: (3, 64, 64)


### The Inefficient Way: Single Image Streaming

In [3]:
# Sequential Processing Pipeline (High Overhead Loop)
# img_1 --> MLP --> prediction_1
# img_2 --> MLP --> prediction_2
# img_3 --> MLP --> prediction_3
# img_4 --> MLP --> prediction_4
# img_5 --> MLP --> prediction_5
# Disadvantage: The hardware must fetch layer weights from memory 5 separate times.

# The Optimized Solution: Pack all images into a unified 4D Batch Tensor
# [N_images, Channels, Height, Width]
batch_of_imgs = np.random.rand(5, 3, 64, 64) 

print(f"Unified batch tensor layout shape [N, C, H, W]: {batch_of_imgs.shape}")
# Execution flow: batch_of_imgs --> MLP --> [pred_1, pred_2, pred_3, pred_4, pred_5]
# Advantage: Weights are fetched exactly once; computations run in parallel.

Unified batch tensor layout shape [N, C, H, W]: (5, 3, 64, 64)


### Simulating the Data Pipeline with a Mock MLP Engine

In [4]:
# Simple dummy MLP class to simulate the forward processing pass
class MockMLP:
    def __init__(self, in_dim, out_dim):
        # Initialize random weights based on input dimensions
        self.weights = np.random.rand(in_dim, out_dim)
        
    def forward(self, x):
        # Expects a 2D tensor input shape: [Batch_Size, Flattened_Features]
        return np.dot(x, self.weights)

# Configuration settings
height, width = 64, 64
flattened_dimensions = 3 * height * width

# Instantiate our dense network architecture
mlp = MockMLP(in_dim=flattened_dimensions, out_dim=1)

# Single sample processing trial: image must be flattened to a 2D row vector
# .reshape(1, -1) converts shape from (3, 64, 64) to (1, 12288)
single_pred = mlp.forward(img_1.reshape(1, -1))
print(f"Single prediction execution output shape: {single_pred.shape}")

Single prediction execution output shape: (1, 1)


### Execution Benchmark: Sequential vs. Batch Forward Passes

In [5]:
print("--- [Method 1] Running 5 Sequential Forward Passes ---")
# Manually reshaping and passing each image vector one after another
y_preds_1 = mlp.forward(img_1.reshape(1, -1))
y_preds_2 = mlp.forward(img_2.reshape(1, -1))
y_preds_3 = mlp.forward(img_3.reshape(1, -1))
y_preds_4 = mlp.forward(img_4.reshape(1, -1))
y_preds_5 = mlp.forward(img_5.reshape(1, -1))

print(f"Prediction 1: {y_preds_1.item():.4f}")
print(f"Prediction 2: {y_preds_2.item():.4f}")
print(f"Prediction 3: {y_preds_3.item():.4f}")
print(f"Prediction 4: {y_preds_4.item():.4f}")
print(f"Prediction 5: {y_preds_5.item():.4f}")


print("\n--- [Method 2] Running a Single Vectorized Batch Forward Pass ---")
# 1. Start with the raw 4D tensor stack: [5, 3, 64, 64]
# 2. Flatten spatial matrices into feature lines while preserving the batch count dimension
# .reshape(5, -1) cleanly collapses dimensions from index 1 onwards -> [5, 12288]
reshaped_batch_tensor = batch_of_imgs.reshape(5, -1)

# MLPs assume 2D matrix inputs. We pass the entire batch simultaneously
batch_predictions = mlp.forward(reshaped_batch_tensor)

print(f"Flattened batch tensor matrix shape passed to MLP: {reshaped_batch_tensor.shape}")
print(f"Batch prediction matrix output shape: {batch_predictions.shape}")
print("Collected Batch Predictions:\n", batch_predictions.flatten())

--- [Method 1] Running 5 Sequential Forward Passes ---
Prediction 1: 3075.0291
Prediction 2: 3079.7854
Prediction 3: 3071.0917
Prediction 4: 3074.5010
Prediction 5: 3104.7728

--- [Method 2] Running a Single Vectorized Batch Forward Pass ---
Flattened batch tensor matrix shape passed to MLP: (5, 12288)
Batch prediction matrix output shape: (5, 1)
Collected Batch Predictions:
 [3076.25270492 3095.9350058  3090.53054462 3058.87168299 3079.06762442]


In [6]:
batch_of_imgs = np.random.rand(5, 3, 64, 64) # 4D Tensor
reshaped_tesor = batch_of_imgs.reshape(5, 3*64*64)
mlp.forward(reshaped_tesor) # MLP Assumes 2D Tensors

array([[3057.41716336],
       [3080.18645202],
       [3084.73794449],
       [3077.66798236],
       [3090.34452845]])